# 02 - Hypothesis Testing

This notebook focuses on conducting A/B Hypothesis Testing to statistically validate or reject key 
hypotheses about risk drivers. These validated hypotheses will form the basis of a new segmentation 
strategy for risk-based pricing. We will quantify "risk" using Claim Frequency and Claim Severity, 
and also analyze "Margin" (TotalPremium - TotalClaims).

Modeling Goals:
Claim Frequency: Proportion of policies with at least one claim.

Claim Severity: The average amount of a claim, given a claim occurred.

Margin: (TotalPremium - TotalClaims).

Null Hypotheses to Test:
H₀: There are no risk differences across provinces.

H₀: There are no risk differences between zip codes.

H₀: There are no significant margin (profit) differences between zip codes.

H₀: There are no significant risk differences between Women and Men.

Methodology:
Data Preparation: Load and prepare the data using functions from src/data_preparation.py.

Metric Calculation: Calculate Claim Frequency, Claim Severity, and Margin using src/statistical_testing.py.

Data Segmentation: For each hypothesis, divide the data into relevant groups (e.g., Province A vs. Province B, Male vs. Female).

Statistical Testing: Conduct appropriate tests (Chi-squared for categorical, t-tests or z-tests for numerical) using functions from src/statistical_testing.py.

P-value < 0.05: Reject the null hypothesis (statistically significant difference).

P-value >= 0.05: Fail to reject the null hypothesis (no statistically significant difference).

Analyze and Report: Document findings and interpret results in a business context.

In [ ]:
# --- Setup and Imports ---
import os
import sys
import pandas as pd
import numpy as np
from scipy.stats import ttest_ind, chi2_contingency
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns

# Add the project root to the Python path for module imports
script_dir = os.path.dirname(os.path.abspath('')) # Get current notebook directory
project_root = os.path.abspath(os.path.join(script_dir, '..'))

if project_root not in sys.path:
    sys.path.insert(0, project_root)
    print(f"Added project root '{project_root}' to sys.path.")
else:
    print(f"Project root '{project_root}' was already in sys.path.")

# Import modules from src
from src.data_preparation import load_raw_data, _convert_comma_to_dot_and_numeric, engineer_features, handle_missing_data
from src.statistical_testing import calculate_risk_metrics, perform_chi_squared_test, perform_t_test, perform_z_test_proportions
from src.config import (
    RAW_DATA_PATH, ALPHA, TOTAL_CLAIMS_COL, TOTAL_PREMIUM_COL, HAS_CLAIM_COL, MARGIN_COL,
    DRIVER_GENDER_COL, PROVINCE_COL, POSTAL_CODE_COL, DATE_FEATURES_CLASSIFICATION
)

# Configure plotting styles
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

# 1. Load and Prepare Data for Hypothesis Testing

In [ ]:
df_raw = load_raw_data(RAW_DATA_PATH)
if df_raw.empty:
    print("Failed to load raw data. Please check RAW_DATA_PATH in src/config.py and data file existence.")
else:
    print("Raw data loaded successfully. Displaying info:")
    df_raw.info()

In [ ]:
# Prepare data for hypothesis testing using the same logic as in run_hypothesis.py
df_ht = df_raw.copy()

# Define all columns that are or should be numeric for initial conversion
potential_numeric_cols_ht = list(set([
    TOTAL_PREMIUM_COL, TOTAL_CLAIMS_COL, 'CustomValueEstimate', 'CapitalOutstanding',
    'SumInsured', 'CalculatedPremiumPerTerm', 'Age', 'VehicleAge',
    'RegistrationYear', 'Cylinders', 'cubiccapacity', 'kilowatts',
    'NumberOfDoors', 'NumberOfVehiclesInFleet'
]))
potential_numeric_cols_ht_in_df = [col for col in potential_numeric_cols_ht if col in df_ht.columns]

# Convert numeric-like columns that might have commas/objects to proper numeric types
df_ht = _convert_comma_to_dot_and_numeric(df_ht, potential_numeric_cols_ht_in_df)

# Engineer features (like date components and vehicle age at transaction)
df_ht = engineer_features(df_ht, DATE_FEATURES_CLASSIFICATION)

# Calculate core risk metrics: HasClaim, ClaimAmountWhenClaimed, Margin.
# This function internally handles making these columns correctly, assuming they're derived
# after initial numerical conversion of TotalClaims/TotalPremium
df_metrics = calculate_risk_metrics(df_ht)

# Define all features for comprehensive missing data handling
all_numeric_cols = [col for col in df_metrics.select_dtypes(include=np.number).columns if col not in [HAS_CLAIM_COL]] # Exclude HAS_CLAIM_COL as it's a binary target
all_categorical_cols = [col for col in df_metrics.select_dtypes(include='object').columns]

# Handle missing data (imputation)
df_prepared_ht = handle_missing_data(df_metrics, all_numeric_cols, all_categorical_cols)

print("Data prepared for Hypothesis Testing. Displaying sample with new metrics:")
display(df_prepared_ht[[HAS_CLAIM_COL, 'ClaimAmountWhenClaimed', MARGIN_COL, PROVINCE_COL, POSTAL_CODE_COL, DRIVER_GENDER_COL]].head())
print("Shape of prepared data:", df_prepared_ht.shape)

# 2. Hypothesis Testing Execution and Interpretation

## Hypothesis 1: H₀: There are no risk differences across provinces

In [ ]:
print("--- Testing H₀: No risk differences across provinces ---")
if PROVINCE_COL in df_prepared_ht.columns:
    # Get top provinces for comparison. Adjust `nlargest` if you want more or specific provinces.
    top_provinces = df_prepared_ht[PROVINCE_COL].value_counts().nlargest(2).index.tolist()
    print(f"Top 2 provinces for comparison: {top_provinces}")

    if len(top_provinces) >= 2:
        province1 = top_provinces[0]
        province2 = top_provinces[1]

        df_prov1 = df_prepared_ht[df_prepared_ht[PROVINCE_COL] == province1].copy()
        df_prov2 = df_prepared_ht[df_prepared_ht[PROVINCE_COL] == province2].copy()

        # Test 1a: Claim Frequency (Proportions)
        claims_prov1 = df_prov1[HAS_CLAIM_COL].sum()
        total_prov1 = len(df_prov1)
        claims_prov2 = df_prov2[HAS_CLAIM_COL].sum()
        total_prov2 = len(df_prov2)
        
        p_value_freq_prov = perform_z_test_proportions(
            claims_prov1, total_prov1, claims_prov2, total_prov2,
            alpha=ALPHA,
            hypothesis_text=f"Claim Frequency difference between {province1} and {province2}"
        )

        # Test 1b: Claim Severity (Mean of ClaimAmountWhenClaimed)
        p_value_sev_prov = perform_t_test(
            df_prov1['ClaimAmountWhenClaimed'].dropna(),
            df_prov2['ClaimAmountWhenClaimed'].dropna(),
            alpha=ALPHA,
            hypothesis_text=f"Claim Severity difference between {province1} and {province2}"
        )
        
        print("\nInterpretation for H1 (Provinces):")
        if (p_value_freq_prov is not np.nan and p_value_freq_prov < ALPHA) or \
           (p_value_sev_prov is not np.nan and p_value_sev_prov < ALPHA):
            print(f"We reject the Null Hypothesis for provinces (at least one p-value < {ALPHA}). This suggests that there ARE statistically significant risk differences across provinces.")
            print(f"- Claim Frequency in {province1}: {claims_prov1/total_prov1:.2%}")
            print(f"- Claim Frequency in {province2}: {claims_prov2/total_prov2:.2%}")
            print(f"- Average Claim Severity in {province1}: ${df_prov1['ClaimAmountWhenClaimed'].mean():,.2f}")
            print(f"- Average Claim Severity in {province2}: ${df_prov2['ClaimAmountWhenClaimed'].mean():,.2f}")
            print("Business Recommendation: Consider incorporating Province as a key factor in risk segmentation and premium adjustments. Further analysis into province-specific factors (e.g., crime rates, driving conditions) may be warranted.")
        else:
            print(f"We fail to reject the Null Hypothesis for provinces (all p-values >= {ALPHA}). This suggests no statistically significant risk differences based on our chosen metrics and provinces.")
            print("Business Recommendation: Based on this analysis, province alone may not be a strong differentiator for risk segmentation.")

    else:
        print(f"Not enough unique provinces ({PROVINCE_COL}) to compare (found {len(top_provinces)}). Skipping province test.")
else:
    print(f"Column '{PROVINCE_COL}' not found for province-based hypothesis testing. Please check config.py and data.")

## Hypothesis 2: H₀: There are no risk differences between zip codes

In [ ]:
print("\n--- Testing H₀: No risk differences between zip codes ---")
if POSTAL_CODE_COL in df_prepared_ht.columns:
    top_zip_codes = df_prepared_ht[POSTAL_CODE_COL].value_counts().nlargest(2).index.tolist()
    print(f"Top 2 zip codes for comparison: {top_zip_codes}")

    if len(top_zip_codes) >= 2:
        zip_code1 = top_zip_codes[0]
        zip_code2 = top_zip_codes[1]

        df_zip1 = df_prepared_ht[df_prepared_ht[POSTAL_CODE_COL] == zip_code1].copy()
        df_zip2 = df_prepared_ht[df_prepared_ht[POSTAL_CODE_COL] == zip_code2].copy()

        # Test 2a: Claim Frequency (Proportions)
        claims_zip1 = df_zip1[HAS_CLAIM_COL].sum()
        total_zip1 = len(df_zip1)
        claims_zip2 = df_zip2[HAS_CLAIM_COL].sum()
        total_zip2 = len(df_zip2)

        p_value_freq_zip = perform_z_test_proportions(
            claims_zip1, total_zip1, claims_zip2, total_zip2,
            alpha=ALPHA,
            hypothesis_text=f"Claim Frequency difference between Zip Code {zip_code1} and {zip_code2}"
        )

        # Test 2b: Claim Severity (Mean of ClaimAmountWhenClaimed)
        p_value_sev_zip = perform_t_test(
            df_zip1['ClaimAmountWhenClaimed'].dropna(),
            df_zip2['ClaimAmountWhenClaimed'].dropna(),
            alpha=ALPHA,
            hypothesis_text=f"Claim Severity difference between Zip Code {zip_code1} and {zip_code2}"
        )
        
        print("\nInterpretation for H2 (Zip Codes - Risk):")
        if (p_value_freq_zip is not np.nan and p_value_freq_zip < ALPHA) or \
           (p_value_sev_zip is not np.nan and p_value_sev_zip < ALPHA):
            print(f"We reject the Null Hypothesis for zip codes (at least one p-value < {ALPHA}). This suggests that there ARE statistically significant risk differences between these top zip codes.")
            print(f"- Claim Frequency in Zip {zip_code1}: {claims_zip1/total_zip1:.2%}")
            print(f"- Claim Frequency in Zip {zip_code2}: {claims_zip2/total_zip2:.2%}")
            print(f"- Average Claim Severity in Zip {zip_code1}: ${df_zip1['ClaimAmountWhenClaimed'].mean():,.2f}")
            print(f"- Average Claim Severity in Zip {zip_code2}: ${df_zip2['ClaimAmountWhenClaimed'].mean():,.2f}\n")
            print("Business Recommendation: Zip code appears to be a significant risk differentiator. Further granular analysis or clustering of zip codes based on risk profiles could lead to more accurate pricing segments.")
        else:
            print(f"We fail to reject the Null Hypothesis for zip codes (all p-values >= {ALPHA}). This suggests no statistically significant risk differences between these top zip codes based on our chosen metrics.\n")
            print("Business Recommendation: While individual zip codes may not show significant differences on their own, more advanced geographic analysis (e.g., geospatial clustering) might reveal patterns.")

    else:
        print(f"Not enough unique zip codes ({POSTAL_CODE_COL}) to compare (found {len(top_zip_codes)}). Skipping zip code risk test.")
else:
    print(f"Column '{POSTAL_CODE_COL}' not found for zip code-based hypothesis testing. Please check config.py and data.")

## Hypothesis 3: H₀: There are no significant margin (profit) differences between zip codes

In [ ]:
print("\n--- Testing H₀: No significant margin (profit) difference between zip codes ---")
if POSTAL_CODE_COL in df_prepared_ht.columns and MARGIN_COL in df_prepared_ht.columns:
    top_zip_codes = df_prepared_ht[POSTAL_CODE_COL].value_counts().nlargest(2).index.tolist()

    if len(top_zip_codes) >= 2:
        zip_code1 = top_zip_codes[0]
        zip_code2 = top_zip_codes[1]

        df_zip1 = df_prepared_ht[df_prepared_ht[POSTAL_CODE_COL] == zip_code1].copy()
        df_zip2 = df_prepared_ht[df_prepared_ht[POSTAL_CODE_COL] == zip_code2].copy()
        
        # Test 3: Margin difference (Mean of Margin)
        p_value_margin_zip = perform_t_test(
            df_zip1[MARGIN_COL].dropna(),
            df_zip2[MARGIN_COL].dropna(),
            alpha=ALPHA,
            hypothesis_text=f"Margin difference between Zip Code {zip_code1} and {zip_code2}"
        )
        
        print("\nInterpretation for H3 (Zip Codes - Margin):")
        if p_value_margin_zip is not np.nan and p_value_margin_zip < ALPHA:
            print(f"We reject the Null Hypothesis for margin differences between zip codes (p < {ALPHA}). This suggests that there ARE statistically significant differences in profit margins between these top zip codes.")
            print(f"- Average Margin in Zip {zip_code1}: ${df_zip1[MARGIN_COL].mean():,.2f}")
            print(f"- Average Margin in Zip {zip_code2}: ${df_zip2[MARGIN_COL].mean():,.2f}\n")
            print("Business Recommendation: Differences in profitability by zip code are significant. This could be due to varying risk profiles not fully captured by current pricing, or operational differences. Investigate drivers of lower margin in specific zip codes for targeted interventions.")
        else:
            print(f"We fail to reject the Null Hypothesis for margin differences between zip codes (p >= {ALPHA}). This suggests no statistically significant difference in profit margins between these top zip codes.\n")
            print("Business Recommendation: Profitability appears consistent across these top zip codes, implying current pricing or risk segmentation is balanced for these areas.")

    else:
        print(f"Not enough unique zip codes ({POSTAL_CODE_COL}) to compare (found {len(top_zip_codes)}). Skipping zip code margin test.")
else:
    print(f"Columns '{POSTAL_CODE_COL}' or '{MARGIN_COL}' not found for margin hypothesis testing. Please check config.py and data.")

## Hypothesis 4: H₀: There are no significant risk differences between Women and Men

In [ ]:
print("\n--- Testing H₀: No significant risk difference between Women and Men ---")
if DRIVER_GENDER_COL in df_prepared_ht.columns:
    # Filter out 'Not specified' or other non-binary values for clean comparison
    gender_categories = ['Male', 'Female']
    df_gender_filtered = df_prepared_ht[df_prepared_ht[DRIVER_GENDER_COL].isin(gender_categories)].copy()

    if gender_categories[0] in df_gender_filtered[DRIVER_GENDER_COL].unique() and \
       gender_categories[1] in df_gender_filtered[DRIVER_GENDER_COL].unique():
        
        df_male = df_gender_filtered[df_gender_filtered[DRIVER_GENDER_COL] == 'Male'].copy()
        df_female = df_gender_filtered[df_gender_filtered[DRIVER_GENDER_COL] == 'Female'].copy()

        # Test 4a: Claim Frequency (Proportions)
        claims_male = df_male[HAS_CLAIM_COL].sum()
        total_male = len(df_male)
        claims_female = df_female[HAS_CLAIM_COL].sum()
        total_female = len(df_female)

        p_value_freq_gender = perform_z_test_proportions(
            claims_male, total_male, claims_female, total_female,
            alpha=ALPHA,
            hypothesis_text=f"Claim Frequency difference between Male and Female Drivers"
        )

        # Test 4b: Claim Severity (Mean of ClaimAmountWhenClaimed)
        p_value_sev_gender = perform_t_test(
            df_male['ClaimAmountWhenClaimed'].dropna(),
            df_female['ClaimAmountWhenClaimed'].dropna(),
            alpha=ALPHA,
            hypothesis_text=f"Claim Severity difference between Male and Female Drivers"
        )
        
        print("\nInterpretation for H4 (Gender - Risk):")
        if (p_value_freq_gender is not np.nan and p_value_freq_gender < ALPHA) or \
           (p_value_sev_gender is not np.nan and p_value_sev_gender < ALPHA):
            print(f"We reject the Null Hypothesis for gender (at least one p-value < {ALPHA}). This suggests that there ARE statistically significant risk differences between men and women drivers.")
            print(f"- Claim Frequency for Male Drivers: {claims_male/total_male:.2%}")
            print(f"- Claim Frequency for Female Drivers: {claims_female/total_female:.2%}")
            print(f"- Average Claim Severity for Male Drivers: ${df_male['ClaimAmountWhenClaimed'].mean():,.2f}")
            print(f"- Average Claim Severity for Female Drivers: ${df_female['ClaimAmountWhenClaimed'].mean():,.2f}\n")
            print("Business Recommendation: Gender appears to be a statistically significant risk factor. This insight can be used to refine pricing and segmentation models, provided it aligns with regulatory and ethical considerations.")
        else:
            print(f"We fail to reject the Null Hypothesis for gender (all p-values >= {ALPHA}). This suggests no statistically significant risk differences between men and women drivers based on our chosen metrics.\n")
            print("Business Recommendation: Gender may not be a strong standalone differentiator for risk in this dataset.")

    else:
        print(f"Not enough distinct gender categories ('Male' and 'Female') found in '{DRIVER_GENDER_COL}' after filtering. Found: {df_prepared_ht[DRIVER_GENDER_COL].unique()}. Skipping gender test.")
else:
    print(f"Column '{DRIVER_GENDER_COL}' not found for gender-based hypothesis testing. Please check config.py and data.")

3. Summary of Hypothesis Testing Results and Business Recommendations
Below is a consolidated summary of the findings from the hypothesis tests. Each interpretation provides a business recommendation based on whether the null hypothesis was rejected or not.

H₀: There are no risk differences across provinces
(Result from Cell 4)

Interpretation: If the null hypothesis was rejected, it indicates that risk metrics (claim frequency or severity) vary significantly across different provinces. For example, if Gauteng showed a higher claim frequency and severity compared to Western Cape.

Business Recommendation: Incorporate province as a significant factor in your risk-based pricing model. This allows for more granular premium adjustments tailored to regional risk profiles. Further investigation into the specific factors driving higher risk in certain provinces (e.g., traffic density, crime rates, weather patterns) could inform targeted risk mitigation strategies.

H₀: There are no risk differences between zip codes
(Result from Cell 5)

Interpretation: If rejected, this means specific zip codes exhibit statistically different risk profiles. Even if provinces show differences, zip codes provide a finer level of granularity. For instance, certain urban zip codes might have significantly higher claim rates or average claim amounts.

Business Recommendation: Leverage zip code information for highly localized risk assessment and pricing. Consider creating geo-demographic segments or using advanced spatial analysis to group similar zip codes. This could lead to fairer and more competitive pricing for customers, and improved profitability for the insurer.

H₀: There are no significant margin (profit) differences between zip codes
(Result from Cell 6)

Interpretation: If rejected, this indicates that the profitability (margin) of policies varies significantly by zip code. A lower margin in certain areas could signal underpricing for the risk incurred, or higher operational costs.

Business Recommendation: Prioritize a review of pricing strategies and operational efficiencies in zip codes with statistically lower margins. This may involve adjusting premiums, refining risk selection criteria, or optimizing claims handling processes in those areas to improve profitability.

H₀: There are no significant risk differences between Women and Men
(Result from Cell 7)

Interpretation: If rejected, this suggests a statistically significant difference in claim frequency or severity between male and female drivers. For example, one gender might have a higher likelihood of claims or incur higher average claim costs.

Business Recommendation: Gender can be considered a statistically significant risk factor for pricing and segmentation, provided its use is compliant with all local laws and regulations regarding non-discriminatory pricing. It can help in creating more precise risk categories and premiums. If ethical or legal restrictions apply, alternative proxies or a focus on behavioral factors correlated with gender might be explored.

Overall Conclusion for Business Strategy:
The outcomes of these hypothesis tests provide crucial insights for developing a dynamic, risk-based pricing system. By identifying which factors (e.g., province, zip code, gender) are statistically significant drivers of risk and profitability, AlphaCare Insurance can:

Refine Segmentation: Create more accurate customer segments based on empirically validated risk differences.

Optimize Premiums: Adjust premiums more precisely to reflect the actual risk and expected profitability associated with individual policies, moving beyond a "one-size-fits-all" approach.

Enhance Competitiveness: Offer more competitive prices to lower-risk segments while ensuring adequate coverage for higher-risk policies.

Improve Profitability: Mitigate losses by accurately pricing risk, leading to healthier margins.

These findings will directly inform the feature selection and model development in Task 4, ensuring that the predictive models are built upon statistically sound insights.